# AI Meeting Notes Analyzer

This notebook implements a LangGraph workflow for extracting topics, a concise summary, assigned actions, and urgency from a meeting transcript. The chat model is accessed through OpenRouter; transcript embeddings are generated locally with Sentence Transformers.

Set `OPENROUTER_API_KEY` in your environment before running the analysis cell. The Streamlit app loads the workflow definition from the tagged runtime cell below.

In [ ]:
from __future__ import annotations

import math
import os
import re
from functools import lru_cache
from pathlib import Path
from typing import Any, Literal, TypedDict

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
load_dotenv(Path.cwd() / ".env")


class TopicsResult(BaseModel):
    topics: list[str] = Field(description="Concise list of distinct topics discussed")


class SummaryResult(BaseModel):
    summary: str = Field(description="A concise meeting summary of 3 to 5 sentences")

    @field_validator("summary")
    @classmethod
    def validate_sentence_count(cls, value: str) -> str:
        sentence_count = len([part for part in re.split(r"(?<=[.!?])\s+", value.strip()) if part])
        if not 3 <= sentence_count <= 5:
            raise ValueError("The meeting summary must contain 3 to 5 sentences.")
        return value


class ActionItem(BaseModel):
    task_description: str = Field(description="Specific task agreed or assigned in the meeting")
    owner: str = Field(
        default="Owner: Not specified",
        description="Responsible person, or exactly 'Owner: Not specified' when the transcript does not name one",
    )

    @field_validator("owner", mode="before")
    @classmethod
    def normalize_missing_owner(cls, value: Any) -> str:
        if not isinstance(value, str) or value.strip().casefold() in {
            "", "not specified", "owner not specified", "unknown", "n/a", "none"
        }:
            return "Owner: Not specified"
        return value.strip()


class ActionsResult(BaseModel):
    action_items: list[ActionItem] = Field(
        description="Explicitly committed or assigned tasks; use an empty list when none exist"
    )


Priority = Literal["High Priority", "Medium Priority", "Low Priority"]


class TaskPriority(BaseModel):
    task_description: str
    priority: Priority


class PriorityResult(BaseModel):
    overall_priority: Priority = Field(description="Overall urgency of the meeting's action items")
    task_priorities: list[TaskPriority] = Field(
        description="Priority for every extracted action item, matching its task description"
    )


class MeetingState(TypedDict, total=False):
    transcript: str
    segments: list[str]
    embedding_vectors: list[list[float]]
    topics: list[str]
    summary: str
    action_items: list[dict[str, str]]
    overall_priority: str
    task_priorities: list[dict[str, str]]
    report: dict[str, Any]


@lru_cache(maxsize=2)
def get_local_embeddings(model_name: str = DEFAULT_EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """Load an embedding model that runs locally on the CPU."""
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


def split_transcript(transcript: str) -> list[str]:
    segments = [part.strip() for part in re.split(r"(?:\r?\n)+|(?<=[.!?])\s+", transcript) if part.strip()]
    return segments or [transcript.strip()]


def build_workflow(
    api_key: str,
    model_name: str = DEFAULT_MODEL,
    embedding_model_name: str = DEFAULT_EMBEDDING_MODEL,
):
    """Build and compile the meeting-analysis LangGraph."""
    if not api_key.strip():
        raise ValueError("An OpenRouter API key is required. Set OPENROUTER_API_KEY or enter it in the app.")

    chat_model = ChatOpenAI(
        model=model_name,
        api_key=api_key,
        base_url=OPENROUTER_BASE_URL,
        temperature=0,
        timeout=90,
    )
    embeddings = get_local_embeddings(embedding_model_name)

    def evidence_for(
        query: str, segments: list[str], segment_vectors: list[list[float]], limit: int = 8
    ) -> str:
        query_vector = embeddings.embed_query(query)
        ranked = []
        for segment, vector in zip(segments, segment_vectors):
            denominator = math.sqrt(sum(value * value for value in query_vector)) * math.sqrt(
                sum(value * value for value in vector)
            )
            similarity = sum(left * right for left, right in zip(query_vector, vector)) / denominator if denominator else 0
            ranked.append((similarity, segment))
        selected = [segment for _, segment in sorted(ranked, reverse=True)[:limit]]
        return "\n".join(f"- {segment}" for segment in selected)

    def input_node(state: MeetingState) -> dict[str, Any]:
        transcript = state.get("transcript", "").strip()
        if not transcript:
            raise ValueError("The meeting transcript cannot be empty.")
        segments = split_transcript(transcript)
        embedding_vectors = embeddings.embed_documents(segments)
        return {"transcript": transcript, "segments": segments, "embedding_vectors": embedding_vectors}

    def topics_node(state: MeetingState) -> dict[str, Any]:
        result = chat_model.with_structured_output(TopicsResult).invoke(
            "Extract the distinct main discussion topics from this meeting. Return short, specific topic labels. "
            "Do not invent topics.\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('main discussion topics and themes', state['segments'], state['embedding_vectors'])}"
        )
        return {"topics": result.topics}

    def summary_node(state: MeetingState) -> dict[str, Any]:
        result = chat_model.with_structured_output(SummaryResult).invoke(
            "Summarize what happened in this meeting in 3 to 5 concise sentences. Preserve important decisions "
            "and context; do not add unsupported facts.\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('meeting purpose, outcomes, and decisions', state['segments'], state['embedding_vectors'])}"
        )
        return {"summary": result.summary}

    def actions_node(state: MeetingState) -> dict[str, Any]:
        result = chat_model.with_structured_output(ActionsResult).invoke(
            "Extract only concrete tasks that someone explicitly commits to or is assigned during the meeting. "
            "Do not treat suggestions, questions, or unassigned possibilities as tasks. For an unknown owner, use "
            "exactly 'Owner: Not specified'. Preserve each task as a concise action.\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('assigned tasks, commitments, and responsible people', state['segments'], state['embedding_vectors'])}"
        )
        return {"action_items": [item.model_dump() for item in result.action_items]}

    def priority_node(state: MeetingState) -> dict[str, Any]:
        result = chat_model.with_structured_output(PriorityResult).invoke(
            "Classify urgency for every action item. Use High Priority for explicit urgent/ASAP language, same-day "
            "deadlines, critical customer or service impact, or deadlines by tomorrow. Use Medium Priority for "
            "near-term deadlines such as this week or Friday, or clear but non-urgent commitments. Use Low Priority "
            "when no urgency or deadline is stated. Return one priority per supplied task and an overall priority "
            "reflecting the most urgent task. Do not invent tasks.\n\n"
            f"Action items:\n{[item['task_description'] + ' — ' + item['owner'] for item in state['action_items']]}\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('task urgency deadlines and priority', state['segments'], state['embedding_vectors'])}"
        )
        if len(result.task_priorities) != len(state['action_items']):
            raise ValueError("The priority agent must return exactly one priority for every action item.")
        return {
            "overall_priority": result.overall_priority,
            "task_priorities": [item.model_dump() for item in result.task_priorities],
        }

    def route_after_actions(state: MeetingState) -> str:
        return "priority" if state.get("action_items") else "finalize"

    def finalize_node(state: MeetingState) -> dict[str, Any]:
        priority_by_task = {
            item["task_description"]: item["priority"] for item in state.get("task_priorities", [])
        }
        actions = [
            {
                **item,
                "priority": priority_by_task.get(
                    item["task_description"], state["task_priorities"][index]["priority"]
                ),
            }
            for index, item in enumerate(state.get("action_items", []))
        ]
        overall_priority = state.get("overall_priority") or "No action items identified in this meeting."
        return {
            "report": {
                "meeting_summary": state["summary"],
                "key_topics": state["topics"],
                "action_items": actions,
                "task_owners": [
                    {"task_description": item["task_description"], "owner": item["owner"]}
                    for item in actions
                ],
                "priority_level": overall_priority,
            }
        }

    graph = StateGraph(MeetingState)
    graph.add_node("input", input_node)
    graph.add_node("topics", topics_node)
    graph.add_node("summary", summary_node)
    graph.add_node("actions", actions_node)
    graph.add_node("priority", priority_node)
    graph.add_node("finalize", finalize_node)
    graph.add_edge(START, "input")
    graph.add_edge("input", "topics")
    graph.add_edge("topics", "summary")
    graph.add_edge("summary", "actions")
    graph.add_conditional_edges("actions", route_after_actions, {"priority": "priority", "finalize": "finalize"})
    graph.add_edge("priority", "finalize")
    graph.add_edge("finalize", END)
    return graph.compile()


## Analyze a transcript

Run the following cell after setting `OPENROUTER_API_KEY`. The first analysis downloads the small embedding model once; embeddings and similarity retrieval run locally.

In [ ]:
from pathlib import Path
import os

transcript_path = Path("Meeting_Transcript.txt")
transcript = transcript_path.read_text(encoding="utf-8")
analyzer = build_workflow(api_key=os.environ["OPENROUTER_API_KEY"])
result = analyzer.invoke({"transcript": transcript})
result["report"]